In [0]:
%sql
use catalog cat_mini;

In [0]:
%sql
create or replace table gold.dim_customers as select 
customer_key , gender , continent from silver.customers;
    
create or replace table gold.dim_stores as select 
store_key,country from silver.stores;
    
create or replace table gold.dim_products as select 
product_key,category from silver.products;
  


In [0]:
%sql
create or replace table gold.fact_sales as select 
  s.order_number,s.line_item,s.order_date,s.delivery_date,
  s.customer_key,s.store_key,s.product_key,s.currency_code,
  s.quantity,p.unit_price_usd,
  (s.quantity * p.unit_price_usd) as revenue,
  datediff(s.delivery_date,s.order_date) as delivery_days,
  case 
    when s.store_key is null then 'Online'
    else 'Store'
  end as channel
from silver.sales s
left join silver.products p
on s.product_key = p.product_key;

In [0]:
%sql
create or replace table gold.cube as 
select * from gold.fact_sales left join gold.dim_customers using(customer_key) left join gold.dim_stores using(store_key) left join gold.dim_products using(product_key);

In [0]:
%sql
update gold.cube
set 
    delivery_days = COALESCE(delivery_days, 0),
    gender = COALESCE(gender, 'Unknown'),
    continent = COALESCE(continent, 'Unknown');